# Aplicação e Análise Comparativa de Métodos Numéricos

## Determinação da altura inicial de uma coluna de água

**Instituição:** Universidade Católica de Pernambuco — UNICAP  
**Curso:** Ciência da Computação  
**Disciplina:** Métodos Numéricos  
**Ano/Semestre:** 2026.2  

Este notebook apresenta a formulação matemática do problema, os cálculos auxiliares, a aplicação dos métodos numéricos, as tabelas de iteração, os gráficos e a análise comparativa dos resultados.

> **Importante:** As implementações dos métodos de **Ponto Fixo, Bisseção, Falsa Posição, Newton-Raphson e Secante não estão neste notebook**. Elas são importadas do pacote Python do projeto, localizado em `src/metodos_numericos/`, conforme solicitado no enunciado da atividade.

# SUMÁRIO

1. [Formulação matemática e definição dos critérios numéricos](#formulacao)
   1. [Formulação do problema](#formulacao-problema)
   2. [Transformação para a forma \(f(H)=0\)](#transformacao)
   3. [Domínio, continuidade e intervalo inicial](#dominio)
   4. [Verificação da unicidade](#unicidade)
   5. [Critérios de parada](#criterios)
   6. [Escolha das tolerâncias](#tolerancias)
   7. [Limite máximo e critério da Bisseção](#limite)
2. [Preparação computacional](#preparacao)
3. [Aplicação dos métodos numéricos](#aplicacao)
   1. [Ponto Fixo](#ponto-fixo)
   2. [Bisseção](#bissecao)
   3. [Falsa Posição](#falsa-posicao)
   4. [Newton-Raphson](#newton)
   5. [Secante](#secante)
4. [Tabela comparativa](#tabela-comparativa)
5. [Gráficos comparativos](#graficos)
6. [Análise comparativa](#analise)
7. [Conclusão](#conclusao)
8. [Referências](#referencias)

<a id="formulacao"></a>
# FORMULAÇÃO MATEMÁTICA E DEFINIÇÃO DOS CRITÉRIOS NUMÉRICOS

<a id="formulacao-problema"></a>

## 1. Formulação do problema

O problema analisado consiste em determinar a altura inicial $H$ da coluna de água necessária para que o escoamento através de um tubo alcance a velocidade de $5\,m/s$ após $2{,}5\,s$. A velocidade é descrita pelo modelo:

$$
v =
\sqrt{2gH}\cdot
\tanh\left[
\left(\frac{\sqrt{2gH}}{2L}\right)t
\right]
$$

em que $v$ representa a velocidade da água, $g$ a aceleração da gravidade, $H$ a altura inicial da coluna de água, $L$ o comprimento do tubo e $t$ o tempo.

Para as condições estabelecidas no problema:

$$
g = 9{,}81\,m/s^2,
\qquad
L = 4\,m,
\qquad
t = 2{,}5\,s,
\qquad
v = 5\,m/s
$$

A equação apresenta caráter não linear, pois a incógnita $H$ aparece simultaneamente em uma raiz quadrada e no argumento de uma função hiperbólica.

Na literatura de análise numérica, a determinação de raízes de equações não lineares é usualmente tratada mediante a formulação $f(x)=0$. Igarashi (1985) discute a solução numérica de equações não lineares nessa forma, incluindo precisão e critérios práticos de término. Sikorski e Woźniakowski (1986) analisam formalmente diferentes critérios de erro para equações escalares não lineares. Brent (1971) aborda a localização de zeros a partir de um intervalo no qual a função muda de sinal.

Considerando essas abordagens, neste trabalho o problema será formulado como uma equação do tipo $f(H)=0$, permitindo aplicar diferentes métodos numéricos à mesma função.

<a id="transformacao"></a>

## 2. Transformação para a forma $f(H)=0$

Substituindo a velocidade requerida, $v=5\,m/s$, no modelo original:

$$
5 =
\sqrt{2gH}\cdot
\tanh\left[
\left(\frac{\sqrt{2gH}}{2L}\right)t
\right]
$$

Reunindo todos os termos em um único membro:

$$
\sqrt{2gH}\cdot
\tanh\left[
\left(\frac{\sqrt{2gH}}{2L}\right)t
\right]-5=0
$$

Define-se:

$$
f(H)=
\sqrt{2gH}\cdot
\tanh\left[
\left(\frac{\sqrt{2gH}}{2L}\right)t
\right]-5
$$

Essa transformação preserva o problema físico original, pois:

$$
f(H)=v(H)-5
$$

Portanto, uma raiz $H^*$ que satisfaça $f(H^*)=0$ corresponde exatamente a uma altura para a qual $v(H^*)=5\,m/s$.

Substituindo $g=9{,}81$, $L=4$ e $t=2{,}5$:

$$
f(H)=
\sqrt{2(9{,}81)H}\cdot
\tanh\left[
\frac{2{,}5\sqrt{2(9{,}81)H}}{2(4)}
\right]-5
$$

Como $2(9{,}81)=19{,}62$ e $2(4)=8$, a função de referência utilizada será:

$$
\boxed{
f(H)=
\sqrt{19{,}62H}\cdot
\tanh\left[
\frac{2{,}5\sqrt{19{,}62H}}{8}
\right]-5
}
$$

<a id="preparacao"></a>
# 2. PREPARAÇÃO COMPUTACIONAL

A célula seguinte importa apenas bibliotecas auxiliares e as funções dos métodos numéricos implementadas no pacote `metodos_numericos`.

Antes de abrir o notebook, o projeto deve ser instalado a partir da raiz do repositório:

```bash
pip install -e .
```

In [2]:
import math
import pandas as pd
import matplotlib.pyplot as plt

from metodos_numericos import (
    ponto_fixo,
    bissecao,
    falsa_posicao,
    newton_raphson,
    secante,
)

print("Ambiente configurado corretamente!")

Ambiente configurado corretamente!


### Constantes e função residual


> **Importante:** A função \(f(H)\) pertence à formulação específica deste problema. Por isso ela é definida no notebook.

In [3]:
g = 9.81
L = 4.0
t = 2.5
v_alvo = 5.0

def f(H):
    return (
        math.sqrt(2 * g * H)
        * math.tanh((math.sqrt(2 * g * H) / (2 * L)) * t)
        - v_alvo
    )

<a id="dominio"></a>

## 3. Domínio, continuidade e escolha do intervalo inicial

Antes da aplicação dos métodos numéricos, é necessário determinar o domínio da função e estabelecer um intervalo adequado para a busca da raiz. Igarashi (1985), em seu estudo sobre problemas práticos relacionados à determinação de raízes de equações não lineares, analisa aspectos envolvidos na resolução numérica de equações formuladas na forma $f(x)=0$. Já Brent (1971), ao propor um algoritmo para determinação de zeros de funções com convergência garantida, parte do princípio de confinamento da raiz em um intervalo no qual a função apresenta mudança de sinal.

Para a função definida neste trabalho,

$$
f(H)=
\sqrt{19{,}62H}\,
\tanh\left(
\frac{2{,}5\sqrt{19{,}62H}}{8}
\right)-5,
$$

a expressão contém o termo:

$$
\sqrt{19{,}62H}
$$

Para que a função permaneça definida no conjunto dos números reais, o radicando deve ser não negativo:

$$
19{,}62H \geq 0
$$

Como $19{,}62>0$, segue que:

$$
\boxed{H\geq0}
$$

Essa restrição matemática também é coerente com a interpretação física do problema, pois $H$ representa a altura inicial de uma coluna de água e, portanto, são considerados valores não negativos.

No domínio $H\geq0$, a função $f(H)$ é composta por funções contínuas. A raiz quadrada é contínua em seu domínio, a função hiperbólica $\tanh$ é contínua para argumentos reais e as operações de composição, multiplicação e subtração preservam a continuidade. Dessa forma, $f(H)$ é contínua no domínio físico considerado.

A continuidade é importante para estabelecer um intervalo que contenha uma raiz. Quando uma função contínua assume valores com sinais opostos nos extremos de um intervalo, isto é,

$$
f(a)\cdot f(b)<0,
$$

a mudança de sinal permite assegurar a existência de pelo menos uma raiz entre esses extremos. Esse princípio é compatível com a abordagem apresentada por Brent (1971), cujo algoritmo considera inicialmente uma função com sinais opostos nos extremos de um intervalo e mantém o confinamento da raiz durante o processo iterativo.

Para a função específica deste trabalho, foram avaliados os valores $H=1$ e $H=2$. Obtém-se aproximadamente:

$$
f(1)\approx-1{,}093721<0
$$

e

$$
f(2)\approx1{,}019273>0
$$

Portanto:

$$
f(1)\cdot f(2)<0
$$

Como $f(H)$ é contínua em $[1,2]$ e apresenta mudança de sinal entre os extremos, existe pelo menos uma raiz no intervalo $(1,2)$. Assim, adota-se:

$$
\boxed{[a,b]=[1,2]}
$$

como intervalo inicial para os métodos que exigem o confinamento da raiz, particularmente os métodos da Bisseção e da Falsa Posição.

É importante destacar que o intervalo $[1,2]$ não é determinado pelas referências bibliográficas. Sua escolha resulta da análise da função específica deste problema, enquanto os trabalhos de Igarashi (1985) e Brent (1971) fornecem fundamentação teórica e metodológica para a determinação numérica de raízes e para o uso de intervalos que confinam uma solução.

In [4]:
a = 1.0
b = 2.0

f_a = f(a)
f_b = f(b)

print(f"f(1) = {f_a:.12f}")
print(f"f(2) = {f_b:.12f}")
print(f"f(1) * f(2) = {f_a * f_b:.12f}")

if f_a * f_b < 0:
    print("Existe mudança de sinal no intervalo [1, 2].")

f(1) = -1.093720547484
f(2) = 1.019273183590
f(1) * f(2) = -1.114800024392
Existe mudança de sinal no intervalo [1, 2].


Os valores obtidos são aproximadamente:

$$
f(1) \approx -1{,}093721 < 0
$$

e

$$
f(2) \approx 1{,}019273 > 0
$$

Consequentemente:

$$
f(1)\cdot f(2) < 0
$$

Como $f$ é contínua em $[1,2]$, existe pelo menos uma raiz $H^*$ no intervalo $(1,2)$. Por essa razão, adota-se:

$$
\boxed{[a,b]=[1,2]}
$$

como intervalo inicial para os métodos que exigem confinamento da raiz, particularmente Bisseção e Falsa Posição.

<a id="unicidade"></a>

## 4. Verificação da unicidade da solução

A mudança de sinal verificada no intervalo $[1,2]$ permite estabelecer a existência de pelo menos uma raiz, mas essa condição, isoladamente, não garante que a raiz seja única. Brent (1971), em seu trabalho sobre um algoritmo com convergência garantida para a determinação de zeros de funções, considera justamente o confinamento de uma raiz em um intervalo no qual a função apresenta mudança de sinal. Esse procedimento assegura a presença de um zero no intervalo sob as condições de continuidade, mas não implica, por si só, a unicidade da solução.

Para investigar a unicidade neste problema, analisa-se o comportamento da derivada de $f(H)$. Pelo resultado clássico de monotonicidade decorrente do Teorema do Valor Médio, se uma função diferenciável possui derivada estritamente positiva em um intervalo, então ela é estritamente crescente nesse intervalo. Assim, demonstrar que $f'(H)>0$ no domínio físico permite verificar que a função não pode assumir o valor zero em dois pontos distintos.

Considerando:

$$
f(H)=
\sqrt{2gH}\,
\tanh\left(
\frac{t\sqrt{2gH}}{2L}
\right)-5,
$$

define-se, para simplificar a expressão:

$$
z=
\frac{t\sqrt{2gH}}{2L}
$$

A derivada da função pode então ser escrita como:

$$
f'(H)=
\frac{g}{\sqrt{2gH}}
\left[
\tanh(z)+
z\,\operatorname{sech}^2(z)
\right]
$$

Para $H>0$, tem-se:

$$
\frac{g}{\sqrt{2gH}}>0
$$

Além disso, como $g>0$, $t>0$, $L>0$ e $H>0$, segue que:

$$
z=
\frac{t\sqrt{2gH}}{2L}>0
$$

Para $z>0$, também são válidas as relações:

$$
\tanh(z)>0
$$

e

$$
\operatorname{sech}^2(z)>0
$$

Consequentemente, todos os termos que compõem $f'(H)$ são positivos para $H>0$, de modo que:

$$
\boxed{f'(H)>0\quad\text{para }H>0}
$$

Portanto, $f(H)$ é estritamente crescente no domínio físico $H>0$. Uma função estritamente crescente não pode assumir o mesmo valor em dois pontos distintos; em particular, não pode possuir duas raízes diferentes nesse domínio.

Como anteriormente foi verificada a existência de uma raiz no intervalo $(1,2)$ por meio da continuidade e da mudança de sinal, e agora foi demonstrado que $f(H)$ é estritamente crescente, conclui-se que essa raiz é única.

Assim:

$$
\boxed{\text{existe uma única raiz }H^*\in(1,2)}
$$

In [5]:
def derivada_f(H):
    s = math.sqrt(2 * g * H)
    z = (t * s) / (2 * L)
    sech2 = 1 / (math.cosh(z) ** 2)

    return (g / s) * (
        math.tanh(z) + z * sech2
    )

for H in [1.0, 1.25, 1.50, 1.75, 2.0]:
    print(f"H = {H:.2f} -> f'(H) = {derivada_f(H):.12f}")

H = 1.00 -> f'(H) = 2.634542982328
H = 1.25 -> f'(H) = 2.317387695041
H = 1.50 -> f'(H) = 2.077065895199
H = 1.75 -> f'(H) = 1.889627928275
H = 2.00 -> f'(H) = 1.739845666313


<a id="criterios"></a>

## 5. Definição dos critérios de parada

A definição de um critério de parada é uma decisão metodológica relevante em algoritmos iterativos. Igarashi (1985) discute a relação entre critérios práticos de término e a precisão efetivamente alcançada. Sikorski e Woźniakowski (1986) distinguem critérios relacionados ao erro e ao resíduo. Rao, Malan e Perot (2018), em outro contexto iterativo, reforçam a importância de critérios que evitem tanto interrupções prematuras quanto trabalho computacional desnecessário.

Neste trabalho serão utilizados simultaneamente dois critérios.

O primeiro mede a variação entre aproximações consecutivas:

$$
\boxed{|H_n-H_{n-1}|<\varepsilon_H}
$$

O segundo mede o resíduo:

$$
\boxed{|f(H_n)|<\varepsilon_f}
$$

Como $f(H)=v(H)-5$, o segundo critério mede diretamente o quanto a velocidade associada à aproximação se afasta da velocidade requerida.

A convergência será declarada somente quando **os dois critérios forem satisfeitos simultaneamente**.

<a id="tolerancias"></a>

## 6. Escolha das tolerâncias

A definição de um critério de parada é uma etapa importante na aplicação de métodos iterativos, pois é necessário estabelecer quando uma aproximação pode ser considerada suficientemente adequada para encerrar o processo numérico.

Igarashi (1985), em seu estudo sobre problemas práticos relacionados à determinação de raízes de equações não lineares, discute critérios de término para processos iterativos e a avaliação da precisão da raiz aproximada obtida. O autor mostra que a decisão de interromper um método numérico está relacionada ao erro que se pretende controlar e às características do problema, não existindo uma regra única que possa ser aplicada de maneira indistinta a todas as situações.

Sikorski e Woźniakowski (1986), ao investigarem diferentes critérios de erro para a solução de equações não lineares, distinguem, entre outros aspectos, critérios relacionados à distância entre a aproximação calculada e uma raiz e critérios baseados no resíduo da função. Essa distinção reforça a importância de especificar qual grandeza está sendo utilizada para avaliar a qualidade de uma aproximação numérica.

Neste trabalho, como a raiz exata não é conhecida durante o processo iterativo, será utilizado como primeiro critério a variação entre duas aproximações sucessivas:

$$
|H_n-H_{n-1}|<\varepsilon_H
$$

Esse critério permite avaliar se as aproximações produzidas pelo método estão apresentando alterações cada vez menores. Entretanto, uma pequena diferença entre duas aproximações sucessivas, considerada isoladamente, não demonstra necessariamente que a equação original esteja sendo satisfeita com o nível de tolerância desejado.

Por esse motivo, será utilizado também o resíduo da equação:

$$
|f(H_n)|<\varepsilon_f
$$

O resíduo mede diretamente o quanto a aproximação $H_n$ satisfaz a equação $f(H)=0$. A utilização conjunta dessas duas verificações permite acompanhar tanto a estabilização das aproximações quanto o atendimento da equação não linear.

Para permitir uma comparação dos cinco métodos numéricos sob as mesmas condições, adotam-se neste trabalho as tolerâncias:

$$
\boxed{\varepsilon_H=10^{-6}\,\text{m}}
$$

e

$$
\boxed{\varepsilon_f=10^{-6}\,\text{m/s}}
$$

Assim, a condição geral de parada utilizada na comparação será:

$$
\boxed{
|H_n-H_{n-1}|<10^{-6}\,\text{m}
\quad\text{e}\quad
|f(H_n)|<10^{-6}\,\text{m/s}
}
$$

As duas condições deverão ser satisfeitas simultaneamente para que o processo iterativo seja considerado convergente.

A escolha numérica de $10^{-6}$ para ambas as tolerâncias é uma decisão metodológica deste trabalho. Esse valor estabelece um limite suficientemente pequeno para comparar os métodos com um mesmo nível de exigência numérica, sem atribuir à tolerância um significado de precisão absoluta da raiz.

Portanto, não se afirma que a utilização de $10^{-6}$ garanta automaticamente seis casas decimais corretas no valor de $H$. Igarashi (1985) destaca justamente a necessidade de distinguir critérios práticos de término da avaliação efetiva da precisão de uma raiz aproximada. Dessa forma, neste trabalho, $10^{-6}$ deve ser interpretado como o limiar adotado para os critérios de convergência e não como uma garantia direta de seis casas decimais exatas.

In [6]:
tolerancia_H = 1e-6
tolerancia_f = 1e-6
max_iter = 100

print(f"Tolerância das iteradas = {tolerancia_H:.0e} m")
print(f"Tolerância residual     = {tolerancia_f:.0e} m/s")
print(f"Máximo de iterações     = {max_iter}")

Tolerância das iteradas = 1e-06 m
Tolerância residual     = 1e-06 m/s
Máximo de iterações     = 100


<a id="limite"></a>

## 7. Limite máximo de iterações e critério adicional da Bisseção

Além dos critérios de convergência definidos anteriormente, será estabelecido um número máximo de iterações como mecanismo de segurança computacional. Igarashi (1985), ao discutir problemas práticos relacionados à determinação de raízes de equações não lineares, aborda a importância dos critérios utilizados para interromper processos iterativos e da avaliação da aproximação obtida. Nesse contexto, é importante distinguir um critério que indica convergência de uma condição utilizada apenas para impedir que o processo iterativo continue indefinidamente.

Neste trabalho, será adotado:

$$
\boxed{N_{\max}=100}
$$

Esse valor constitui uma decisão metodológica própria deste trabalho e funciona como limite de segurança para a execução dos algoritmos. Portanto, atingir $N_{\max}$ não significa que o método convergiu. A convergência somente será considerada quando os critérios de parada definidos anteriormente forem satisfeitos. Caso o número máximo de iterações seja atingido antes disso, o método será registrado como não convergente dentro das condições estabelecidas para o experimento.

---

### 7.1. Critério adicional para o Método da Bisseção

O Método da Bisseção possui uma característica que permite estabelecer diretamente um limite para o erro da aproximação. O método parte de um intervalo $[a,b]$ que contém uma raiz e, a cada iteração, divide esse intervalo ao meio, preservando o subintervalo no qual ocorre a mudança de sinal. Brent (1971), em seu trabalho sobre um algoritmo com convergência garantida para determinação de zeros de funções, utiliza a Bisseção como procedimento seguro para preservar o confinamento da raiz quando etapas de interpolação não são adequadas.

Se, na iteração $n$, a raiz $H^*$ está confinada no intervalo $[a_n,b_n]$ e $H_n$ é escolhido como seu ponto médio, então:

$$
H_n=\frac{a_n+b_n}{2}
$$

Como a raiz permanece dentro do intervalo, a maior distância possível entre o ponto médio e qualquer ponto pertencente a esse intervalo corresponde à metade de seu comprimento. Portanto:

$$
\boxed{
|H^*-H_n|
\leq
\frac{b_n-a_n}{2}
}
$$

Essa propriedade fornece à Bisseção uma estimativa direta do limite superior para o erro da aproximação, independentemente do valor exato de $H^*$.

Por esse motivo, além dos critérios gerais utilizados para comparar os cinco métodos, será monitorada, especificamente para a Bisseção, a condição:

$$
\boxed{
\frac{b_n-a_n}{2}<10^{-6}\,\text{m}
}
$$

O valor $10^{-6}\,\text{m}$ corresponde à tolerância para a variável $H$ definida anteriormente. Esse critério adicional permite relacionar diretamente o comprimento do intervalo de confinamento ao limite teórico do erro do ponto médio.

---

### 7.2. Cálculo do número teórico de iterações mediante logaritmo

Uma vantagem do Método da Bisseção é que a redução do intervalo ocorre de maneira previsível. Como cada iteração reduz pela metade o comprimento do intervalo anterior, após $n$ bisseções o comprimento do intervalo é:

$$
\frac{b-a}{2^n}
$$

Consequentemente, utilizando o ponto médio como aproximação, o limite para o erro é:

$$
|H^*-H_n|
\leq
\frac{b-a}{2^{n+1}}
$$

Essa redução sistemática do intervalo explica a convergência garantida da Bisseção quando as hipóteses de continuidade e mudança de sinal são satisfeitas e permite determinar previamente quantas divisões são necessárias para que o limite do erro seja inferior a uma tolerância estabelecida.

Para o intervalo adotado neste trabalho,

$$
[a,b]=[1,2],
$$

tem-se:

$$
b-a=1
$$

Deseja-se que o limite teórico do erro seja inferior à tolerância adotada para $H$:

$$
\frac{b-a}{2^{n+1}}<10^{-6}
$$

Multiplicando por $2^{n+1}$ e dividindo por $10^{-6}$:

$$
2^{n+1}>
\frac{b-a}{10^{-6}}
$$

Aplicando o logaritmo na base $2$:

$$
n+1>
\log_2\left(
\frac{b-a}{10^{-6}}
\right)
$$

Portanto:

$$
\boxed{
n>
\log_2\left(
\frac{b-a}{10^{-6}}
\right)-1
}
$$

Para $b-a=1$:

$$
n>
\log_2(10^6)-1
$$

Como:

$$
\log_2(10^6)\approx19{,}931569,
$$

obtém-se:

$$
n>18{,}931569
$$

Assim, considerando essa indexação, o menor número inteiro que satisfaz a desigualdade é:

$$
\boxed{n=19}
$$

Esse resultado representa uma estimativa teórica baseada exclusivamente na redução do intervalo pela Bisseção. Ele não deve ser confundido com o número de iterações que será efetivamente observado quando o algoritmo for executado utilizando simultaneamente todos os critérios de parada estabelecidos neste trabalho.

A célula seguinte realiza somente o cálculo teórico do número de iterações por meio do logaritmo. A implementação do Método da Bisseção permanece no pacote `metodos_numericos` e não é realizada nesta célula do notebook.

In [ ]:
epsilon_bissecao = 1e-6

valor_log = math.log2((b - a) / epsilon_bissecao)
limite_n = valor_log - 1

n_teorico = math.floor(limite_n) + 1

while (b - a) / (2 ** (n_teorico + 1)) >= epsilon_bissecao:
    n_teorico += 1

print(f"log2((b-a)/epsilon) = {valor_log:.12f}")
print(f"n > {limite_n:.12f}")
print(f"Menor inteiro que satisfaz a desigualdade: n = {n_teorico}")
print(
    "Limite do erro correspondente = "
    f"{(b-a)/(2**(n_teorico+1)):.12e} m"
)

log2((b-a)/epsilon) = 19.931568569324
n > 18.931568569324
Menor inteiro que satisfaz a desigualdade: n = 19
Limite do erro correspondente = 9.536743164062e-07 m


> **Importante:** A implementação do algoritmo não é realizada diretamente neste notebook. Conforme a organização adotada para o projeto, os Métodos estão implementado no módulo `bissecao.py`, `falsa_posicao.py`, `newton_raphson.py`, `ponto_fixo.py` e `secante.py`, localizados no pacote `src/metodos_numericos`. Nesta seção, o algoritmo é apenas chamado para resolver a função $f(H)$ do problema, utilizando o intervalo inicial e os critérios de convergência previamente definidos.

<a id="aplicacao"></a>

# 3. APLICAÇÃO DOS MÉTODOS NUMÉRICOS

Nesta seção, os algoritmos são apenas **chamados**. Suas implementações permanecem em `src/metodos_numericos/`.

Para manter a comparação coerente, todos os métodos utilizam:

$$
\varepsilon_H=10^{-6},
\qquad
\varepsilon_f=10^{-6},
\qquad
N_{\max}=100
$$

<a id="ponto-fixo"></a>

## 3.1 Método do Ponto Fixo

Para aplicar o Método do Ponto Fixo, a equação $f(H)=0$ deve ser reescrita na forma:

$$
H=\phi(H)
$$

A função de iteração $\phi(H)$ utilizada neste trabalho é obtida a partir da própria equação do problema. Partindo de:

$$
f(H)=
\sqrt{19{,}62H}\,
\tanh\left(
\frac{2{,}5\sqrt{19{,}62H}}{8}
\right)-5=0
$$

isolando o termo correspondente à velocidade, obtém-se:

$$
\sqrt{19{,}62H}\,
\tanh\left(
\frac{2{,}5\sqrt{19{,}62H}}{8}
\right)=5
$$

Dividindo os dois membros pelo termo hiperbólico:

$$
\sqrt{19{,}62H}
=
\frac{5}{
\tanh\left(
\frac{2{,}5\sqrt{19{,}62H}}{8}
\right)
}
$$

Elevando os dois membros ao quadrado:

$$
19{,}62H
=
\frac{25}{
\tanh^2\left(
\frac{2{,}5\sqrt{19{,}62H}}{8}
\right)
}
$$

Por fim, dividindo os dois membros por $19{,}62$, a variável $H$ fica isolada:

$$
H=
\frac{25}{
19{,}62
\tanh^2\left(
\frac{2{,}5\sqrt{19{,}62H}}{8}
\right)
}
$$

Dessa forma, comparando a expressão obtida com $H=\phi(H)$, define-se a função de iteração:

$$
\boxed{
\phi(H)=
\frac{25}{
19{,}62
\tanh^2\left(
\frac{2{,}5\sqrt{19{,}62H}}{8}
\right)
}
}
$$

Assim, o processo iterativo do Método do Ponto Fixo é dado por:

$$
H_{n+1}=\phi(H_n)
$$

ou, explicitamente:

$$
H_{n+1}
=
\frac{25}{
19{,}62
\tanh^2\left(
\frac{2{,}5\sqrt{19{,}62H_n}}{8}
\right)
}
$$

Como aproximação inicial, será utilizado o ponto médio do intervalo $[1,2]$, previamente estabelecido pela análise da mudança de sinal da função:

$$
H_0=\frac{1+2}{2}=1{,}5
$$

Portanto:

$$
\boxed{H_0=1{,}5}
$$

In [8]:
def phi(H):
    argumento = (2.5 * math.sqrt(19.62 * H)) / 8

    return 25 / (
        19.62
        * math.tanh(argumento) ** 2
    )

resultado_pf = ponto_fixo(
    phi=phi,
    f=f,
    x0=1.5,
    tolerancia_x=tolerancia_H,
    tolerancia_f=tolerancia_f,
    max_iter=max_iter,
)

resultado_pf

{'raiz': 1.465894515445561,
 'convergiu': True,
 'iteracoes': 9,
 'erro': 3.837320974309222e-07,
 'residuo': 1.5382663942631325e-07,
 'historico': [{'iteracao': 1,
   'x_anterior': 1.5,
   'x_atual': 1.458097784293705,
   'erro': 0.04190221570629493,
   'residuo': 0.016449093385046076},
  {'iteracao': 2,
   'x_anterior': 1.458097784293705,
   'x_atual': 1.4677390900328726,
   'erro': 0.00964130573916755,
   'residuo': 0.0038836150604311115},
  {'iteracao': 3,
   'x_anterior': 1.4677390900328726,
   'x_atual': 1.4654616902854805,
   'erro': 0.0022773997473921614,
   'residuo': 0.0009119009108884413},
  {'iteracao': 4,
   'x_anterior': 1.4654616902854805,
   'x_atual': 1.4659963788960406,
   'erro': 0.0005346886105601634,
   'residuo': 0.000214397983326009},
  {'iteracao': 5,
   'x_anterior': 1.4659963788960406,
   'x_atual': 1.4658706643151045,
   'erro': 0.00012571458093613153,
   'residuo': 5.039202679490984e-05},
  {'iteracao': 6,
   'x_anterior': 1.4658706643151045,
   'x_atual': 1.

In [9]:
tabela_pf = pd.DataFrame(resultado_pf["historico"])
tabela_pf

,iteracao,x_anterior,x_atual,erro,residuo
0,1,1.500000,1.458098,4.190222e-02,1.644909e-02
1,2,1.458098,1.467739,9.641306e-03,3.883615e-03
2,3,1.467739,1.465462,2.277400e-03,9.119009e-04
3,4,1.465462,1.465996,5.346886e-04,2.143980e-04
4,5,1.465996,1.465871,1.257146e-04,5.039203e-05
5,6,1.465871,1.465900,2.954772e-05,1.184497e-05
6,7,1.465900,1.465893,6.945393e-06,2.784190e-06
7,8,1.465893,1.465895,1.632531e-06,6.544333e-07
8,9,1.465895,1.465895,3.837321e-07,1.538266e-07


<a id="bissecao"></a>

## 3.2 Método da Bisseção

O Método da Bisseção é um método de confinamento para determinação de raízes de equações não lineares. Sua aplicação parte de um intervalo $[a,b]$ no qual uma função contínua apresenta valores com sinais opostos nos extremos.

Para o problema deste trabalho, foi verificado anteriormente que:

$$
f(1)\approx-1{,}093721<0
$$

e

$$
f(2)\approx1{,}019273>0
$$

Portanto:

$$
f(1)\cdot f(2)<0
$$

Como $f(H)$ é contínua no intervalo considerado, a mudança de sinal garante a existência de pelo menos uma raiz entre os extremos. Além disso, a análise da derivada realizada anteriormente mostrou que $f(H)$ é estritamente crescente para $H>0$, permitindo concluir que a raiz existente nesse intervalo é única.

Dessa forma, será utilizado como intervalo inicial:

$$
\boxed{[a,b]=[1,2]}
$$

A escolha específica dos valores $1$ e $2$ decorre da análise da função deste problema e não de uma exigência do Método da Bisseção.

Em cada iteração, o método calcula o ponto médio do intervalo corrente:

$$
H_n=\frac{a_n+b_n}{2}
$$

e avalia $f(H_n)$. A partir do sinal obtido, é selecionado o subintervalo no qual a mudança de sinal permanece. Dessa forma, a raiz continua confinada entre os extremos do novo intervalo.

Como o comprimento do intervalo é reduzido pela metade a cada iteração, a Bisseção também permite estabelecer um limite para o erro associado ao ponto médio:

$$
|H^*-H_n|
\leq
\frac{b_n-a_n}{2}
$$

Esse critério adicional já foi discutido anteriormente na seção referente ao limite máximo de iterações e ao cálculo teórico da Bisseção.


In [ ]:
## chamar a função aqui para aplicar o metodo

In [ ]:
# printar a tabela  do resultados das iteracoes aqu


<a id="falsa-posicao"></a>

## 3.3 Método da Falsa Posição

O Método da Falsa Posição é um método de determinação de raízes que utiliza um intervalo inicial $[a,b]$ no qual a função apresenta sinais opostos nos extremos. Assim como ocorre na Bisseção, essa condição permite manter a raiz confinada em um intervalo durante o processo iterativo.

Para o problema deste trabalho, foi verificado anteriormente que:

$$
f(1)\approx-1{,}093721<0
$$

e

$$
f(2)\approx1{,}019273>0
$$

Portanto:

$$
f(1)\cdot f(2)<0
$$

Como $f(H)$ é contínua no intervalo considerado, a mudança de sinal garante a existência de pelo menos uma raiz entre os extremos. Dessa forma, será utilizado como intervalo inicial:

$$
\boxed{[a,b]=[1,2]}
$$

A escolha específica dos valores $1$ e $2$ decorre da análise da função deste problema e não de uma exigência do Método da Falsa Posição.

Diferentemente da Bisseção, que utiliza o ponto médio do intervalo como nova aproximação, a Falsa Posição utiliza uma interpolação linear entre os pontos $(a_n,f(a_n))$ e $(b_n,f(b_n))$. A interseção da reta determinada por esses dois pontos com o eixo horizontal fornece a nova aproximação da raiz.

Essa aproximação pode ser calculada por:

$$
H_n=
\frac{
a_nf(b_n)-b_nf(a_n)
}{
f(b_n)-f(a_n)
}
$$

ou, de forma equivalente:

$$
H_n=
a_n-
f(a_n)
\frac{
b_n-a_n
}{
f(b_n)-f(a_n)
}
$$

Após o cálculo de $H_n$, o sinal de $f(H_n)$ é utilizado para determinar qual extremo do intervalo deve ser substituído. O novo intervalo é escolhido de forma que a mudança de sinal seja preservada:

$$
f(a_n)\cdot f(b_n)<0
$$

Dessa maneira, a raiz permanece confinada durante as iterações.

Portanto, embora Bisseção e Falsa Posição utilizem o mesmo intervalo inicial neste trabalho, os métodos diferem na forma de determinar a nova aproximação: a Bisseção utiliza o ponto médio do intervalo, enquanto a Falsa Posição utiliza a interseção de uma reta secante com o eixo horizontal.

In [ ]:
## chamar a função aqui para aplicar o metodo

In [ ]:
# printar a tabela  do resultados das iteracoes aqui

<a id="newton"></a>

## 3.4 Método de Newton-Raphson

O Método de Newton-Raphson é um método iterativo para determinação de raízes de equações não lineares. A partir de uma aproximação inicial $H_0$, as aproximações seguintes são calculadas por:

$$
H_{n+1}
=
H_n-
\frac{f(H_n)}{f'(H_n)}
$$

Diferentemente dos métodos de confinamento, como a Bisseção, o Método de Newton-Raphson necessita de uma aproximação inicial para iniciar o processo iterativo. A escolha desse valor pode influenciar o comportamento e a convergência do método.

Neste trabalho, já foi estabelecido anteriormente que a raiz procurada está no intervalo:

$$
[1,2]
$$

Como não se deseja utilizar o valor previamente conhecido da solução para favorecer a escolha da aproximação inicial, será adotado o ponto médio desse intervalo como critério objetivo para definir $H_0$:

$$
H_0=\frac{a+b}{2}
$$

Substituindo $a=1$ e $b=2$:

$$
H_0=\frac{1+2}{2}=1{,}5
$$

Portanto:

$$
\boxed{H_0=1{,}5}
$$

A escolha de $H_0=1{,}5$ é, portanto, uma decisão metodológica deste trabalho baseada no intervalo previamente identificado como contendo a raiz, e não no conhecimento antecipado do valor da solução.

In [ ]:
## chamar a função aqui para aplicar o metodo

In [ ]:
# printar a tabela  do resultados das iteracoes aqui

<a id="secante"></a>

## 3.5 Método da Secante

O Método da Secante é um método iterativo para determinação de raízes de equações não lineares que, diferentemente do Método de Newton-Raphson, não necessita do cálculo analítico da derivada. Em seu lugar, utiliza a inclinação da reta secante determinada por duas aproximações consecutivas.

O processo iterativo é dado por:

$$
H_{n+1}
=
H_n-
f(H_n)
\frac{H_n-H_{n-1}}
{f(H_n)-f(H_{n-1})}
$$

Como o cálculo de uma nova aproximação depende de dois valores anteriores, o Método da Secante necessita de duas aproximações iniciais, $H_0$ e $H_1$.

Neste trabalho, já foi estabelecido anteriormente que a raiz procurada está no intervalo $[1,2]$, pois:

$$
f(1)\approx-1{,}093721<0
$$

e

$$
f(2)\approx1{,}019273>0
$$

Portanto:

$$
f(1)\cdot f(2)<0
$$

Como critério objetivo para a escolha das duas aproximações iniciais, serão utilizados os próprios extremos do intervalo previamente identificado como contendo a raiz:

$$
\boxed{H_0=1}
$$

e

$$
\boxed{H_1=2}
$$

Essa escolha é uma decisão metodológica deste trabalho e evita utilizar conhecimento antecipado do valor da solução para selecionar aproximações artificialmente próximas da raiz.

É importante observar que, embora $H_0$ e $H_1$ tenham sido escolhidos a partir do intervalo $[1,2]$, o Método da Secante não é um método de confinamento como a Bisseção. Assim, suas aproximações posteriores não são obrigadas a permanecer dentro desse intervalo.

In [ ]:
## chamar a função aqui para aplicar o metodo

In [ ]:
# printar a tabela  do resultados das iteracoes aqui

<a id="tabela-comparativa"></a>
# 4. TABELA COMPARATIVA DOS RESULTADOS

Para comparar os métodos sob os mesmos critérios, será construída uma tabela com:

- aproximação final;
- número de iterações;
- erro entre as duas últimas aproximações;
- resíduo final;
- indicação de convergência.

In [ ]:
# printar a tabela  comparativas dos metodos

<a id="graficos"></a>
# 5. GRÁFICOS COMPARATIVOS

Os gráficos abaixo permitem comparar visualmente a eficiência dos métodos em termos de número de iterações, erro final e resíduo final.

In [ ]:
# plotar grafico  comparativo dos metodos: nun interaçoes

In [ ]:
# plotar grafico  comparativo dos metodos: Comparação do erro final

In [ ]:

# plotar grafico  comparativas dos metodos: Comparação do resíduo final

### Convergência das aproximações

O gráfico seguinte utiliza os históricos retornados pelos métodos. Como os nomes das colunas de aproximação podem variar entre os algoritmos, a célula procura automaticamente uma coluna adequada para cada histórico.

In [ ]:
# plotar grafico  comparativas dos metodos: Convergência das aproximações

<a id="analise"></a>
# 6. ANÁLISE COMPARATIVA

A comparação deve considerar não apenas a aproximação final obtida, mas também o custo iterativo e as características de cada método.

AQUI TEMOS QUE DESCREVER A ANALASE COMPARATIVAS ENTRE OS METODOS

In [ ]:


## tabela para mostras o metodo com melhor desempenho

<a id="conclusao"></a>
# 7. CONCLUSÃO

AQUI TEMOS QUE FAZER AS CONCLUSOES FINAIS

<a id="referencias"></a>

# REFERÊNCIAS

BRENT, R. P. An algorithm with guaranteed convergence for finding a zero of a function. *The Computer Journal*, Oxford, v. 14, n. 4, p. 422–425, 1971. DOI: 10.1093/comjnl/14.4.422. Disponível em: https://doi.org/10.1093/comjnl/14.4.422. Acesso em: 11 set. 2026.

CORDERO, A.; TORREGROSA, J. R. A class of multi-point iterative methods for nonlinear equations. *Applied Mathematics and Computation*, Amsterdam, v. 197, n. 1, p. 337–344, 2008. DOI: 10.1016/j.amc.2007.07.047. Disponível em: https://doi.org/10.1016/j.amc.2007.07.047. Acesso em: 11 set. 2026.

HUANG, N.; MA, C. Convergence analysis and numerical study of a fixed-point iterative method for solving systems of nonlinear equations. *The Scientific World Journal*, [S. l.], v. 2014, Article ID 789459, p. 1–10, 2014. DOI: 10.1155/2014/789459. Disponível em: https://doi.org/10.1155/2014/789459. Acesso em: 11 set. 2026.

IGARASHI, M. Practical problems arising for finding roots of nonlinear equations. *Applied Numerical Mathematics*, Amsterdam, v. 1, n. 5, p. 433–455, 1985. DOI: 10.1016/0168-9274(85)90005-4. Disponível em: https://doi.org/10.1016/0168-9274(85)90005-4. Acesso em: 11 set. 2026.

IGARASHI, M. Practical stopping rule for finding roots of non-linear equations. *Journal of Computational and Applied Mathematics*, Amsterdam, v. 12–13, p. 371–380, 1985. DOI: 10.1016/0377-0427(85)90031-7. Disponível em: https://doi.org/10.1016/0377-0427(85)90031-7. Acesso em: 11 set. 2026.

RAO, K.; MALAN, P.; PEROT, J. B. A stopping criterion for the iterative solution of partial differential equations. *Journal of Computational Physics*, Amsterdam, v. 352, p. 265–284, 2018. DOI: 10.1016/j.jcp.2017.09.033. Disponível em: https://doi.org/10.1016/j.jcp.2017.09.033. Acesso em: 11 set. 2026.

SCHERZER, O. Convergence criteria of iterative methods based on Landweber iteration for solving nonlinear problems. *Journal of Mathematical Analysis and Applications*, [S. l.], v. 194, n. 3, p. 911–933, 1995. DOI: 10.1006/jmaa.1995.1335. Disponível em: https://doi.org/10.1006/jmaa.1995.1335. Acesso em: 11 set. 2026.

SIKORSKI, K.; WOŹNIAKOWSKI, H. For which error criteria can we solve nonlinear equations? *Journal of Complexity*, [S. l.], v. 2, n. 2, p. 163–178, 1986. DOI: 10.1016/0885-064X(86)90017-8. Disponível em: https://doi.org/10.1016/0885-064X(86)90017-8. Acesso em: 11 set. 2026.